# Week 10 demo: TF-IDF, embeddings and LLM labels

The code from the slides, in order.

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/macss-berkeley/compss-211a/main/docs/interactives/data/week08/aita_replies.csv"
comments = pd.read_csv(url)
comments["verdict"] = comments["body"].str.upper().str.extract(r"^(NTA|YTA)\b")[0]
comments = comments.dropna(subset=["verdict"]).reset_index(drop=True)
len(comments)  # 1035

In [ ]:
verdict_words = r"\b(NTA|YTA|ESH|NAH|INFO)\b"
comments["text"] = comments["body"].str.replace(verdict_words, "", regex=True, case=False)
comments["text"] = comments["text"].str.lstrip(" .,:;!-–—")

examples = [204, 232, 75]  # the three comments on the slides
comments.loc[examples, ["verdict", "text"]]

## TF-IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words="english", min_df=15)  # words in at least 15 comments
weights = pd.DataFrame(tfidf.fit_transform(comments["text"]).toarray(),
                       columns=tfidf.get_feature_names_out())
weights.shape  # (1035, 220)

In [ ]:
weights.loc[examples, ["thinking", "bad", "know", "taking", "stay", "job"]].round(2)

In [ ]:
by_verdict = weights.groupby(comments["verdict"]).mean().T
by_verdict["ratio"] = by_verdict["YTA"] / by_verdict["NTA"]
by_verdict.sort_values("ratio")

In [ ]:
by_ratio = by_verdict.sort_values("ratio")
selected = pd.concat([by_ratio.head(6), by_ratio.tail(6)])
selected[["NTA", "YTA"]].plot.barh(figsize=(6, 5), xlabel="average TF-IDF weight", ylabel="")

## Embeddings

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")  # downloads about 90 MB the first time
embeddings = model.encode(comments["text"].tolist(), normalize_embeddings=True)
embeddings.shape  # (1035, 384)

In [ ]:
pd.DataFrame(embeddings[examples, :4], index=examples).round(2)  # the first 4 of 384 numbers

In [ ]:
query = model.encode(["Your partner is controlling you."], normalize_embeddings=True)[0]
comments["similarity"] = embeddings @ query  # how close each comment is to the sentence
comments.sort_values("similarity", ascending=False).head(4)[["similarity", "verdict", "text"]]

## An LLM label

Needs `GEMINI_API_KEY` in your `.env` file. Three calls.

In [ ]:
import os

from dotenv import load_dotenv
from google import genai

load_dotenv()
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

In [ ]:
question = ("Does this comment say the poster was in the wrong? "
            "Answer yes or no, then give a one-sentence reason.")

def label(text):
    response = client.models.generate_content(model="gemini-3.5-flash-lite",
                                              contents=question + "\n\nComment: " + text)
    return response.text

comments.loc[examples, "text"].map(label)  # 3 comments, 3 calls

## Sentiment with VADER

In [ ]:
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer

nltk.download("vader_lexicon", quiet=True)  # the word list, downloaded once
sia = SentimentIntensityAnalyzer()
comments["vader"] = comments["text"].map(lambda text: sia.polarity_scores(text)["compound"])  # from -1 to +1
comments.loc[examples, ["verdict", "vader", "text"]]

## Agreement between two labelers

In [ ]:
from sklearn.metrics import cohen_kappa_score

labels_a = ["yes", "yes", "no", "no", "no", "no", "no", "no", "no", "no"]
labels_b = ["yes", "no", "yes", "no", "no", "no", "no", "no", "no", "no"]
agreement = (pd.Series(labels_a) == pd.Series(labels_b)).mean()
agreement, cohen_kappa_score(labels_a, labels_b)  # (0.8, 0.375)